# Classification for Malicious URL's Detection

This notebook aims to detect malicious URLs before they are used in an application .

Steps : 
1. Problem definition
2. Data
3. Features
4. Modelling
6. Experimentation


# Preblem definition
In an applpication,
> Given an image or live screen , we will extract url's from it and use our model to predict whether they are : Benign , Defacement , Phishing , Malware .

# Data 

The data is from kaggle . For more information you can visit the following link : 
>https://www.kaggle.com/datasets/sid321axn/malicious-urls-dataset?select=malicious_phish.csv

# Features

The csv file contain 2 columns : 
* URL : here is the path 
* Type : can be Bening , Defacement , Phishing , Malware

# Modelling

In [1]:
import pandas as pd 
import numpy as np
import re
from urllib.parse import urlparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report
import joblib

In [2]:
# Extract data with pandas 
data = pd.read_csv("csv_datasets/malicious_phish.csv")
data.head()

,url,type
0,br-icloud.com.br,phishing
1,mp3raid.com/music/krizz_kaliko.html,benign
2,bopsecrets.org/rexroth/cr/1.htm,benign
3,http://www.garage-pirenne.be/index.php?option=...,defacement
4,http://adventure-nicaragua.net/index.php?optio...,defacement


In [3]:
# Informations about the dataset
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 651191 entries, 0 to 651190
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   url     651191 non-null  object
 1   type    651191 non-null  object
dtypes: object(2)
memory usage: 9.9+ MB


In [4]:
# Check for null values
data.isna().sum()

url     0
type    0
dtype: int64

In [5]:
data.describe()

,url,type
count,651191,651191
unique,641119,4
top,http://style.org.hc360.com/css/detail/mysite/s...,benign
freq,180,428103


In [6]:
# Normalize data for training
data_type = data["type"].to_numpy()
unique_type = np.unique(data_type)

bool_type = [each_type == unique_type for each_type in data_type] 
bool_type[:2]

[array([False, False, False,  True]), array([ True, False, False, False])]

In [7]:
print(bool_type[0])
bool_type[0].argmax()
bool_type[0].astype(int)

[False False False  True]


array([0, 0, 0, 1])

In [8]:
int_type = [t.astype(int) for t in bool_type]
int_type[:3]

[array([0, 0, 0, 1]), array([1, 0, 0, 0]), array([1, 0, 0, 0])]

In [9]:
# Create a function to normalize URL's
def normalize_url(url) :
    # transform to lowercase
    url = url.lower()
    # remove protocol (http / https) - use re from py -> regular expression
    url = re.sub(r'^https?:\/\/','',url)
    # remove "www."
    url = re.sub(r'^www\.','',url)
    # remove slashes
    url = url.rstrip('/')

    return url

In [10]:
data_url = data["url"]
normalized_urls = [normalize_url(url) for url in data_url]

In [11]:
normalized_urls[:2]

['br-icloud.com.br', 'mp3raid.com/music/krizz_kaliko.html']

In [12]:
vectorize_urls = TfidfVectorizer(analyzer='char_wb' , ngram_range=(3,5))

In [13]:
# Create variables X & y for split 
X = vectorize_urls.fit_transform(data_url[:4000])
y = int_type[:4000]

In [14]:
# Split into train and test data
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.2 , random_state=42)

In [15]:
# Train model 
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train , y_train)
print(f"KNN Accuracy : {knn.score(X_train,y_train)*100:.2f}%")

KNN Accuracy : 93.12%


In [16]:
y_pred = knn.predict(X_test)
print(classification_report(y_test , y_pred , zero_division=0))

              precision    recall  f1-score   support

           0       0.93      0.97      0.95       592
           1       0.91      0.90      0.91       138
           2       0.92      0.48      0.63        25
           3       0.89      0.38      0.53        45

   micro avg       0.93      0.91      0.92       800
   macro avg       0.92      0.68      0.75       800
weighted avg       0.93      0.91      0.91       800
 samples avg       0.91      0.91      0.91       800



In [17]:
# Save the model
joblib.dump(knn , "malicious_url_detection.pkl")
# Save vectorizer
joblib.dump(vectorize_urls , "vectorizer.pkl")

['vectorizer.pkl']

In [18]:
y_pred

array([[1, 0, 0, 0],
       [1, 0, 0, 0],
       [1, 0, 0, 0],
       ...,
       [1, 0, 0, 0],
       [1, 0, 0, 0],
       [1, 0, 0, 0]], shape=(800, 4))

In [19]:
y_pred[0]

array([1, 0, 0, 0])

In [28]:
predicted = data_type[np.argmax(y_pred)]
print(predicted)

phishing
